In [1]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# Initialize system configurations

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Verify BEFORE creating the session

print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")\
    .getOrCreate()



PYSPARK_PYTHON: c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe
HADOOP_HOME: C:\hadoop


c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [4]:
df_read_dim_business_data  = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_business")

In [5]:
df_read_business_data.columns

['address',
 'attributes',
 'business_id',
 'categories',
 'city',
 'hours',
 'is_open',
 'latitude',
 'longitude',
 'name',
 'postal_code',
 'review_count',
 'stars',
 'state']

In [6]:
df_hours = df_read_business_data.select(F.col("hours"))

In [8]:
df_hours.show(truncate=False)

+------------------------------------------------------------------------------------+
|hours                                                                               |
+------------------------------------------------------------------------------------+
|NULL                                                                                |
|{8:0-18:30, 0:0-0:0, 8:0-14:0, NULL, 8:0-18:30, 8:0-18:30, 8:0-18:30}               |
|{8:0-23:0, 8:0-22:0, 8:0-23:0, 8:0-22:0, 8:0-22:0, 8:0-22:0, 8:0-22:0}              |
|{7:0-21:0, 7:0-20:0, 7:0-21:0, 7:0-21:0, 7:0-20:0, 7:0-20:0, 7:0-20:0}              |
|{12:0-22:0, NULL, 12:0-22:0, 12:0-18:0, 16:0-22:0, NULL, 14:0-22:0}                 |
|{9:0-0:0, 0:0-0:0, 9:0-22:0, 8:0-22:0, 6:0-22:0, 6:0-22:0, 6:0-22:0}                |
|{10:0-18:0, 0:0-0:0, 10:0-18:0, 12:0-18:0, 10:0-18:0, 10:0-18:0, 10:0-18:0}         |
|{9:0-17:0, 9:0-17:0, NULL, NULL, 9:0-17:0, 9:0-17:0, 9:0-17:0}                      |
|NULL                                      

In [9]:
df_hours.show(truncate=False)

+------------------------------------------------------------------------------------+
|hours                                                                               |
+------------------------------------------------------------------------------------+
|NULL                                                                                |
|{8:0-18:30, 0:0-0:0, 8:0-14:0, NULL, 8:0-18:30, 8:0-18:30, 8:0-18:30}               |
|{8:0-23:0, 8:0-22:0, 8:0-23:0, 8:0-22:0, 8:0-22:0, 8:0-22:0, 8:0-22:0}              |
|{7:0-21:0, 7:0-20:0, 7:0-21:0, 7:0-21:0, 7:0-20:0, 7:0-20:0, 7:0-20:0}              |
|{12:0-22:0, NULL, 12:0-22:0, 12:0-18:0, 16:0-22:0, NULL, 14:0-22:0}                 |
|{9:0-0:0, 0:0-0:0, 9:0-22:0, 8:0-22:0, 6:0-22:0, 6:0-22:0, 6:0-22:0}                |
|{10:0-18:0, 0:0-0:0, 10:0-18:0, 12:0-18:0, 10:0-18:0, 10:0-18:0, 10:0-18:0}         |
|{9:0-17:0, 9:0-17:0, NULL, NULL, 9:0-17:0, 9:0-17:0, 9:0-17:0}                      |
|NULL                                      

In [10]:
# This function is used to get subcategory names from the 'categories' column

@F.udf(returnType=ArrayType(StringType())) # Define DataFrame schema
def get_category_name(text):
    if text is None :
      return ["Not Specified"]   

    
    try:
        splitted = str(text).split(",")
    except:
       return ["Not Specified"]   
    llt=[] 
    lit=str(text).split(",")
    for i in lit:
      llt.append(i)
 #  st = text.split(",")
   # for i in st:
   #    lit.append("unknow")
    return(list(lit))

# Step 1: apply UDF, extract fields, then CACHE + materialize 

df_udf = df_hours.withColumn("subcategory_name_list", get_category_name(F.col("hours")))


#df_udf.show(truncate=False)
df_udf.show(truncate=False,n=2000)  # Materialization happens at this stage

# Step 2: explode AFTER the UDF stage is fully materialized



+-------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hours                                                                                      |subcategory_name_list                                                                                                                                                        |
+-------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|NULL                                                                                       |[Not Specified]                                                                                        

In [32]:
def parse_schedule_entry(text):
   if text is None:
      return ["Not Specified"],["Not Specified"]  ,["Not Specified"]     

   operating_hours = str(text).split(",")
   start  = []
   end = []
   day =[]
   for schedule in operating_hours:
      start_hour="" 
      end_hour="" 
      count = 0

      clean_day_name = schedule.lower().strip()

      if "none" in clean_day_name or  "null" in clean_day_name or (
      "friday" not in clean_day_name and  
      "saturday" not in clean_day_name and 
      "sunday" not in clean_day_name and 
      "monday" not in clean_day_name and
      "tuesday" not in clean_day_name and
      "wednesday" not in clean_day_name and
      "thursday" not in clean_day_name  ) :
         continue
            
      for slot in schedule:
         if  slot == "-":
            count = 1 

         if slot == ":" and count == 1 :
            end_hour = end_hour + ":"

         if slot == ":" and count == 0 :
            start_hour = start_hour + ":"

         if  slot.isdigit() and count == 1:  
            end_hour=end_hour + slot

         if slot.isdigit() and count == 0:

            start_hour=start_hour + slot

      if not start_hour  or not end_hour :
         continue
   
      if "friday" in clean_day_name and "friday" not in day:
         day.append("friday")
        
      if "saturday" in clean_day_name and "saturday" not in day:
         day.append("saturday")
        
      if "sunday" in clean_day_name and "sunday" not in day:
         day.append("sunday")  
        
      if "monday" in clean_day_name and "monday" not in day:
         day.append("monday")
        
      if "tuesday" in clean_day_name and "tuesday" not in day:
         day.append("tuesday")
           
      if "wednesday" in clean_day_name and "wednesday" not in day:
         day.append("wednesday")
          
      if "thursday" in clean_day_name and "thursday" not in day:
         day.append("thursday")
           
      try:
         date_object = datetime.strptime(start_hour, '%H:%M').time()
         start_time_12h = date_object.strftime("%I:%M %p")
      except ValueError:
         start_time_12h = "Time Not Set"

      try:
         date_object = datetime.strptime(end_hour, '%H:%M').time()
         end_time_12h = date_object.strftime("%I:%M %p")
      except ValueError:
         end_time_12h = "Time Not Set"

      if start_time_12h in  "Time Not Set" or end_time_12h in "Time Not Set" :
        start_time_12h = "Time Not Set" 
        end_time_12h = "Time Not Set"  
   
      start.append(start_time_12h)
      end.append(end_time_12h )
      



   if "friday" not in day:
      day.append("friday")
      start.append("Time Not Set")
      end.append("Time Not Set")

   if "saturday" not in day:
      day.append("saturday")
      start.append("Time Not Set")
      end.append("Time Not Set")

   if "sunday" not in day:
      day.append("sunday")  
      start.append("Time Not Set")
      end.append("Time Not Set")

   if "monday" not in day:
      day.append("monday")
      start.append("Time Not Set")
      end.append("Time Not Set")
   if "tuesday" not in day:
      day.append("tuesday")   
      start.append("Time Not Set")
      end.append("Time Not Set")
   if "wednesday" not in day:
      day.append("wednesday")  
      start.append("Time Not Set")
      end.append("Time Not Set")
   if "thursday" not in day:
      day.append("thursday") 
      start.append("Time Not Set")
      end.append("Time Not Set")
   return day , start , end   


schema = StructType([
    StructField("day_name", ArrayType(StringType()), True), 
    StructField("shift_start_hour", ArrayType(StringType()), True),
    StructField("shift_end_hour", ArrayType(StringType()), True),  
])

parse_schedule_entry_udf = F.udf(parse_schedule_entry, schema)

df_elite = (
    df_hours.withColumn("new_eilte", parse_schedule_entry_udf(F.col("hours")))
      .withColumn("day_name", F.col("new_eilte.day_name"))
      .withColumn("shift_start_hour", F.col("new_eilte.shift_start_hour"))
      .withColumn("shift_end_hour", (F.col("new_eilte.shift_end_hour")))
).drop(F.col("new_eilte")).cache()  

df_elite.show(n=1000,truncate=False)

+-------------------------------------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------+
|hours                                                                                      |day_name                                                        |shift_start_hour                                                                              |shift_end_hour                                                                                |
+-------------------------------------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------------------------------------+-------------------------------------------------

In [36]:

df_explode = (
    df_elite.withColumn("zipped", F.arrays_zip("day_name", "shift_start_hour", "shift_end_hour"))
      .withColumn("zipped", F.explode(F.col("zipped")))
      .select(
          #"business_id",  # or whatever your identifying columns are
          F.col("zipped.day_name").alias("day_name"),
          F.col("zipped.shift_start_hour").alias("shift_start_hour"),
          F.col("zipped.shift_end_hour").alias("shift_end_hour"),
      ).distinct()
)
df_explode.show(truncate=False)

+---------+----------------+--------------+
|day_name |shift_start_hour|shift_end_hour|
+---------+----------------+--------------+
|sunday   |07:00 AM        |09:00 PM      |
|thursday |07:00 AM        |08:00 PM      |
|tuesday  |10:00 AM        |06:00 PM      |
|monday   |09:00 AM        |05:00 PM      |
|thursday |09:00 AM        |05:00 PM      |
|thursday |06:00 AM        |04:00 PM      |
|sunday   |10:00 AM        |08:00 PM      |
|tuesday  |09:30 AM        |09:30 PM      |
|wednesday|11:00 AM        |02:00 PM      |
|friday   |10:00 AM        |08:00 PM      |
|wednesday|08:00 AM        |05:00 PM      |
|thursday |11:30 AM        |08:30 PM      |
|monday   |08:00 AM        |06:00 PM      |
|friday   |10:00 AM        |09:00 PM      |
|monday   |11:00 AM        |09:00 PM      |
|saturday |11:00 AM        |09:00 PM      |
|saturday |12:00 PM        |09:00 PM      |
|monday   |08:00 AM        |07:00 PM      |
|wednesday|04:00 PM        |10:00 PM      |
|monday   |10:00 AM        |12:0

In [37]:
df_explode.orderBy(F.col("shift_start_hour").desc()).show(n=1000,truncate=False)

+-------------+----------------+--------------+
|day_name     |shift_start_hour|shift_end_hour|
+-------------+----------------+--------------+
|friday       |Time Not Set    |Time Not Set  |
|wednesday    |Time Not Set    |Time Not Set  |
|tuesday      |Time Not Set    |Time Not Set  |
|thursday     |Time Not Set    |Time Not Set  |
|monday       |Time Not Set    |Time Not Set  |
|saturday     |Time Not Set    |Time Not Set  |
|sunday       |Time Not Set    |Time Not Set  |
|Not Specified|Not Specified   |Not Specified |
|wednesday    |12:45 PM        |03:30 PM      |
|tuesday      |12:45 PM        |03:30 PM      |
|tuesday      |12:45 PM        |08:00 PM      |
|monday       |12:45 PM        |03:30 PM      |
|friday       |12:45 PM        |03:30 PM      |
|thursday     |12:45 PM        |03:30 PM      |
|thursday     |12:45 PM        |11:00 PM      |
|sunday       |12:45 PM        |04:00 PM      |
|saturday     |12:45 PM        |06:00 PM      |
|thursday     |12:45 PM        |06:00 PM

In [ ]:
df_explode.show(n=1000,truncate=False)

In [31]:
day_columns = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

stack_expr = "stack(7, " + ", ".join(
    [f"'{d}', hours.{d}" for d in day_columns]
) + ") as (day_name, day_range)"

df_unpivoted = df_hours.select("business_id", F.expr(stack_expr))

time_pattern = r"(\d{1,2}):(\d{1,2})-(\d{1,2}):(\d{1,2})"

df_elite = (
    df_unpivoted
    .withColumn("has_valid_time", F.col("day_range").rlike(time_pattern))
    .withColumn("start_hour_raw", F.regexp_extract("day_range", time_pattern, 1).cast("int"))
    .withColumn("start_min_raw", F.regexp_extract("day_range", time_pattern, 2).cast("int"))
    .withColumn("end_hour_raw", F.regexp_extract("day_range", time_pattern, 3).cast("int"))
    .withColumn("end_min_raw", F.regexp_extract("day_range", time_pattern, 4).cast("int"))
    .withColumn("shift_start_hour",
        F.when(~F.col("has_valid_time"), F.lit("Time Not Set"))
         .otherwise(F.date_format(
             F.make_timestamp(F.lit(2000), F.lit(1), F.lit(1), F.col("start_hour_raw"), F.col("start_min_raw"), F.lit(0)),
             "hh:mm a")))
    .withColumn("shift_end_hour",
        F.when(~F.col("has_valid_time"), F.lit("Time Not Set"))
         .otherwise(F.date_format(
             F.make_timestamp(F.lit(2000), F.lit(1), F.lit(1), F.col("end_hour_raw"), F.col("end_min_raw"), F.lit(0)),
             "hh:mm a")))
    .drop("has_valid_time", "start_hour_raw", "start_min_raw", "end_hour_raw", "end_min_raw", "day_range")
)

df_elite.show(n=1000,truncate=False)

+----------------------+---------+----------------+--------------+
|business_id           |day_name |shift_start_hour|shift_end_hour|
+----------------------+---------+----------------+--------------+
|Pns2l4eNsfO8kk83dixA6A|Monday   |NULL            |NULL          |
|Pns2l4eNsfO8kk83dixA6A|Tuesday  |NULL            |NULL          |
|Pns2l4eNsfO8kk83dixA6A|Wednesday|NULL            |NULL          |
|Pns2l4eNsfO8kk83dixA6A|Thursday |NULL            |NULL          |
|Pns2l4eNsfO8kk83dixA6A|Friday   |NULL            |NULL          |
|Pns2l4eNsfO8kk83dixA6A|Saturday |NULL            |NULL          |
|Pns2l4eNsfO8kk83dixA6A|Sunday   |NULL            |NULL          |
|mpf3x-BjTdTEA3yCZrAYPw|Monday   |12:00 AM        |12:00 AM      |
|mpf3x-BjTdTEA3yCZrAYPw|Tuesday  |08:00 AM        |06:30 PM      |
|mpf3x-BjTdTEA3yCZrAYPw|Wednesday|08:00 AM        |06:30 PM      |
|mpf3x-BjTdTEA3yCZrAYPw|Thursday |08:00 AM        |06:30 PM      |
|mpf3x-BjTdTEA3yCZrAYPw|Friday   |08:00 AM        |06:30 PM   

In [ ]:
df_explode =  df_elite.withColumn("Day_name", F.explode(F.col("Day_name")))\
               .withColumn("shift_start_hour", F.explode(F.col("shift_start_hour")))\
               .withColumn("shift_end_hour", F.explode((F.col("shift_end_hour"))))

In [ ]:
df_explode.show()

In [ ]:
# This function is used to get subcategory names from the 'categories' column

#@F.udf(returnType=ArrayType(StringType())) # Define DataFrame schema
def get_category_name(text):
    if text is None:
       return "Not Specified",["Not Specified"]  ,["Not Specified"]  
    st = str(text).split(",")
    name=" "
    start_date =[]
    end_date = []
    day=[]
    for i in st:
        stt = str(i).lower().strip()
        if ( "Friday" not in str(text) ) and ( "friday" not in day ) :
            day.append("friday")
            start_date.append("Not specified")
            end_date.append("Not specified")
            continue

        if ("Saturday"  not in str(text) ) and ("saturday"  not in day):
            day.append("saturday")
            start_date.append("Not specified")
            end_date.append("Not specified")
            continue      


        if ( "Sunday"  not in str(text)  ) and  ("sunday"  not in day):
            day.append("sunday") 
            start_date.append("Not specified")
            end_date.append("Not specified")
            continue      

        if ("Monday"  not in str(text) ) and  ("monday"  not in day):
            day.append("monday")
            start_date.append("Not specified")
            end_date.append("Not specified")
            continue      

        if ("Tuesday"  not in str(text) ) and  ("tuesday"  not in day):
            day.append("tuesday")
            start_date.append("Not specified")
            end_date.append("Not specified")
            continue      
      
        if ("Wednesday"  not in str(text) ) and  ("wednesday"  not in day) :
            day.append("wednesday")
            start_date.append("Not specified")
            end_date.append("Not specified")
            continue      

        if ("Thursday"  not in str(text)) and  ("thursday"  not in day):
            day.append("thursday")
            start_date.append("Not specified")
            end_date.append("Not specified")
            continue      


        stt =str(i).strip()
        sttyy=stt.split("'")
        
        for j in sttyy:
            st1 = str(j).split("-")  
            lower_day = j.strip().lower()

            if "friday" in lower_day:
              name = "friday"

            if "tuesday" in lower_day:
              name = "tuesday"           

            if "saturday" in lower_day:
              name = "saturday"                       

            if "sunday"   in lower_day:
              name = "sunday"    

            if "monday"   in lower_day:
                name = "monday"  
           
            if "tuesday"   in lower_day:
               name = "tuesday"

            if "wednesday"   in lower_day:
               name = "wednesday"  

            if "thursday"   in lower_day:
               name = "wednesday"  

            if len(st1)>1:
                count = 0  
                for t in st1:
                    num_1 = t[-2:].strip() 
                    num_2 = t[-1:].strip()
                    num_4 = t[:2].strip()
                    num_3 = t[:1].strip() 
                    time_1 = 0
                    time_2 = 0
                        
                    if num_2.isnumeric():
                        time_1 = num_2
                        
                        if num_1.isnumeric():
                                time_1 = num_1
                    if num_3.isnumeric():
                        time_2 = num_3

                        if num_4.isnumeric():
                                time_2 = num_4  
                    if int(time_2)>=0 and int(time_2) <=11:
                        if int(time_2)== 0: 
                            time_2 = "12"
                        str_1 = time_2 + " : " + time_1 + " AM"
                    
                        if count == 0 :
                              start_date.append(str_1)
                              count = count +1 
                              continue  
                        elif  count == 1:
                              end_date.append(str_1)
                            

                    if int(time_2)>=12 and int(time_2)<=23:

                        if int(time_2) > 12 :
                            time_2 = str(int(time_2) - 12)  
                        str_1 = time_2 + " : " + time_1 + " PM"
                
                        if count == 0 :
                              start_date.append(str_1)
                              count = count +1 
                              continue   
                        elif  count == 1:
                              end_date.append(str_1)
        day.append(name)                          

    return  day,start_date,end_date       

schema = StructType([
    StructField("Day_name", StringType(), True), 
    StructField("shift_start_hour", ArrayType(StringType()), True),
    StructField("shift_end_hour", ArrayType(StringType()), True),  # <-- fixed
])

get_category_name_udf = F.udf(get_category_name, schema)

df_elite = (
    df_hours.withColumn("new_eilte", get_category_name_udf(F.col("hours")))
      .withColumn("Day_name", F.col("new_eilte.Day_name"))
      .withColumn("shift_start_hour", F.col("new_eilte.shift_start_hour"))
      .withColumn("shift_end_hour", (F.col("new_eilte.shift_end_hour")))
).drop(F.col("new_eilte"))


# Step 1: apply UDF, extract fields, then CACHE + materialize 

#df_udf = df_hours.withColumn("subcategory_name_list", get_category_name(F.col("hours")))


#df_udf.show(truncate=False)
df_elite.show(truncate=False,n=1000)  # Materialization happens at this stage

# Step 2: explode AFTER the UDF stage is fully materialized



+-------------------------------------------------------------------------------------------+-----------------------------------------------------------------+------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hours                                                                                      |Day_name                                                         |Categorie_name                                                                      |last_elite_year                                                                                                                                                         |
+-------------------------------------------------------------------------------------------+-----------------------------------------------------------------+-------------

In [ ]:
def parse_schedule_entry(text):
   if text is None:
      return ["Not Specified"],["Not Specified"]  ,["Not Specified"]     
   st = str(text).split(",")
   start  = [ ]
   end = [ ]
   day =[]
   for i in st:
      hour="" 
      minute="" 
      hour1="" 
      minute1="" 
      count = 0

      if "none" in i.lower().strip() or  "null" in i.lower().strip() or (
      "friday" not in i.lower().strip() and  
      "saturday" not in i.lower().strip() and 
      "sunday" not in i.lower().strip() and 
      "monday" not in i.lower().strip() and
      "tuesday" not in i.lower().strip() and
      "wednesday" not in i.lower().strip() and
      "thursday" not in i.lower().strip()  ) :
         continue
            
      for j in i:
         if  j == "-":
            count = 1 
         if j == ":" and count == 1 :
            hour1 = hour1 + ":"

         if j == ":" and count == 0 :
            hour = hour + ":"

         if  j.isdigit() and count == 1:  
            hour1=hour1 +j

         if j.isdigit() and count == 0:

            hour=hour +j


      if not hour  or not hour1 :
         continue
      
      if "friday" in i.lower().strip() and "friday" not in day:
         day.append("friday")
         
      if "saturday" in i.lower().strip() and "saturday" not in day:
         day.append("saturday")
         
      if "sunday" in i.lower().strip() and "sunday" not in day:
         day.append("sunday")  
         
      if "monday" in i.lower().strip() and "monday" not in day:
         day.append("monday")
         
      if "tuesday" in i.lower().strip() and "tuesday" not in day:
         day.append("tuesday")
            
      if "wednesday" in i.lower().strip() and "wednesday" not in day:
         day.append("wednesday")
         
      if "thursday" in i.lower().strip() and "thursday" not in day:
         day.append("thursday")
            

      date_object = datetime.strptime(hour, '%H:%M').time()
      s0 = date_object.strftime("%I:%M %p")

      date_object = datetime.strptime(hour1, '%H:%M').time()
      s1 = date_object.strftime("%I:%M %p")
   
      start.append(s0)
      end.append(s1  )




   if "friday" not in day:
         day.append("friday")
         start.append("Time Not Set")
         end.append("Time Not Set")

   if "saturday" not in day:
         day.append("saturday")
         start.append("Time Not Set")
         end.append("Time Not Set")

   if "sunday" not in day:
      day.append("sunday")  
      start.append("Time Not Set")
      end.append("Time Not Set")

   if "monday" not in day:
      day.append("monday")
      start.append("Time Not Set")
      end.append("Time Not Set")
   if "tuesday" not in day:
      day.append("tuesday")   
      start.append("Time Not Set")
      end.append("Time Not Set")
   if "wednesday" not in day:
      day.append("wednesday")  
      start.append("Time Not Set")
      end.append("Time Not Set")
   if "thursday" not in day:
      day.append("thursday") 
      start.append("Time Not Set")
      end.append("Time Not Set")

   return day , start , end      

schema = StructType([
    StructField("day_name", ArrayType(StringType()), True), 
    StructField("shift_start_hour", ArrayType(StringType()), True),
    StructField("shift_end_hour", ArrayType(StringType()), True),  
])

parse_schedule_entry_udf = F.udf(parse_schedule_entry, schema)

df_elite = (
    df_hours.withColumn("new_eilte", parse_schedule_entry_udf(F.col("hours")))
      .withColumn("day_name", F.col("new_eilte.day_name"))
      .withColumn("shift_start_hour", F.col("new_eilte.shift_start_hour"))
      .withColumn("shift_end_hour", (F.col("new_eilte.shift_end_hour")))
).drop(F.col("new_eilte")).cache()

df_elite.show(truncate=False)      

NameError: name 'day' is not defined

In [ ]:
df = df_subcategory_name.select(F.col("subcategory_name"))

In [ ]:
df.filter(F.col("subcategory_name").isin("0:0-0:0") ).show(n=df.count(),truncate=False)

+----------------+
|subcategory_name|
+----------------+
+----------------+



In [ ]:
st = df_hours.select(F.col("hours")).first()

In [ ]:
type(st)

pyspark.sql.types.Row